# FloodShift-04F — 3-Class Performance Push

**Goal:** one controlled successor to the frozen `04E FloodShift-FINAL-TerraMind-RIFT` model, aimed at improving the same joint three-class Sen1Floods11 task:

- `0 = non_flood / background`
- `1 = permanent_water`
- `2 = temporary_flood`
- `255 = ignore`

### Frozen reference (04E)
- Validation, 4-way TTA: **mIoU 0.7791**, permanent **0.8244**, temporary **0.5416**
- Geographic test: **mIoU 0.7162**, permanent **0.6183**, temporary **0.5675**

### Predeclared 04F validation promotion gate
The frozen test is not touched unless 04F reaches all of:
- **mIoU >= 0.790**
- **temporary-flood IoU >= 0.570**
- **permanent-water IoU >= 0.820**

### What changes — and what does NOT
**Unchanged:** dataset split, target semantics, frozen preprocessing, TerraMind-RIFT base architecture, 256×256 evaluation protocol, 4-way TTA decision rule.

**04F changes:**
1. Warm-start from the sanitized 04E checkpoint rather than retraining from scratch.
2. Fixed **class-aware stochastic crop oversampling** for temporary flood, permanent water, and their difficult interface. This is intentionally preferred over pure OHEM because published evidence shows fixed oversampling can be more stable than adaptive hard-mining under extreme imbalance.
3. A **zero-initialized hierarchical residual refiner** that separately learns (a) water-vs-background correction and (b) temporary-vs-permanent correction. At initialization, 04F is *exactly* the 04E model.
4. A targeted loss: multiclass CE + Lovász + conditional permanent-vs-temporary CE + water-union Dice + class-specific focal-Tversky + boundary CE.
5. Conservative staged tuning: refiner-only → selected decoder/fusion modules → at most the final TerraMind transformer block. No full backbone unfreeze.
6. Loss is computed in FP32 and non-finite batches are skipped rather than poisoning epoch statistics.

### Research basis
- GEOID-Flood (ECCV 2026 TerraBytes) explicitly uses **background / permanent water / flooded water**, finds optical-SAR fusion to be the strongest route for transient flooding, and uses TerraMind-B as a practical foundation backbone. https://arxiv.org/abs/2608.02315
- Banerjee & Daou (2026 preprint) use the same three-class Sen1Floods11 objective and report their strongest flooded-land IoU with SegFormer-B2. https://arxiv.org/abs/2606.16302
- Bai et al. (Remote Sensing 2021) support S1/S2 fusion and focal-style treatment of flood-water imbalance on Sen1Floods11. https://doi.org/10.3390/rs13112220
- Boundary-aware objectives can improve remote-sensing segmentation IoU where precise delineation matters. https://arxiv.org/abs/1905.07852
- Focal-Tversky / asymmetric compound losses are well established for strongly imbalanced segmentation.

**Scientific rule:** this notebook may tune on **train + validation only**. The geographic test set is locked by default.


In [ ]:
# ============================================================
# CELL: P8-C00 — KAGGLE-COMPAT TERRATORCH INSTALL
# ============================================================
# Proven compatible path from 04E.
# IMPORTANT: on a fresh Kaggle image, pip may change NumPy to 1.26.4.
# If the next cell reports a NumPy/Pandas ABI error, restart the Python
# kernel ONCE, do NOT rerun this cell, then continue from P8-C01.

import sys, subprocess

print("Installing TerraTorch 1.1.1 (04E-proven compatibility path)...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", "--no-cache-dir", "terratorch==1.1.1"
])
print("P8-C00 COMPLETE ✅")
print("If NumPy changed, restart the Python kernel once before P8-C01.")


In [ ]:
# ============================================================
# CELL: P8-C01 — ENVIRONMENT + REPRODUCIBILITY
# ============================================================
import os
# Use one T4 deliberately even if Kaggle exposes two.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import sys, gc, re, json, time, math, random, copy, hashlib, zipfile
from pathlib import Path

import numpy as np
import scipy
from scipy import ndimage
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import rasterio
from rasterio.windows import Window

import terratorch
from terratorch.registry import BACKBONE_REGISTRY

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()
torch.backends.cudnn.benchmark = True

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/FloodShift_04F_3Class_Push")
CHECKPOINT_ROOT = OUTPUT_ROOT / "checkpoints"
HISTORY_ROOT = OUTPUT_ROOT / "history"
METRIC_ROOT = OUTPUT_ROOT / "metrics"
CALIBRATION_ROOT = OUTPUT_ROOT / "calibration"
BOOTSTRAP_ROOT = OUTPUT_ROOT / "bootstrap"
for p in [OUTPUT_ROOT, CHECKPOINT_ROOT, HISTORY_ROOT, METRIC_ROOT, CALIBRATION_ROOT, BOOTSTRAP_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print("=" * 112)
print("P8-C01 — 04F ENVIRONMENT")
print("=" * 112)
print("Python      :", sys.version.split()[0])
print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("SciPy       :", scipy.__version__)
print("PyTorch     :", torch.__version__)
print("TerraTorch  :", getattr(terratorch, "__version__", "unknown"))
print("Device      :", DEVICE)
print("Visible GPUs:", torch.cuda.device_count())
if torch.cuda.is_available(): print("GPU         :", torch.cuda.get_device_name(0))
print("AMP         :", AMP_ENABLED)
print("Output root :", OUTPUT_ROOT)


In [ ]:
# ============================================================
# CELL: P8-C02 — LOCATE FROZEN PREPROCESSING + 04E CHECKPOINT
# ============================================================
_file_cache = {}

def locate_by_saved_path(saved_path_or_name, required=True):
    saved = Path(str(saved_path_or_name))
    filename = saved.name
    cache_key = str(saved_path_or_name)
    if cache_key in _file_cache:
        return _file_cache[cache_key]
    roots = [INPUT_ROOT, BOOTSTRAP_ROOT]
    matches = []
    for root in roots:
        if root.exists(): matches.extend(root.rglob(filename))
    if not matches:
        if required:
            raise FileNotFoundError(f"Could not locate '{filename}' under /kaggle/input or bootstrap.")
        return None
    selected = sorted(matches, key=lambda p: (len(str(p)), str(p)))[0]
    _file_cache[cache_key] = selected
    return selected

# If user attached the previously downloaded ZIP as a Kaggle dataset, extract it automatically.
archive_matches = list(INPUT_ROOT.rglob("FloodShift_FINAL_ARCHIVE.zip"))
if archive_matches:
    archive = sorted(archive_matches, key=lambda p: len(str(p)))[0]
    print("Extracting attached 04E archive:", archive)
    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(BOOTSTRAP_ROOT)

FINAL_MANIFEST_PATH = locate_by_saved_path("floodshift_preprocessing_manifest_v1.csv")
GLOBAL_LOCK_PATH = locate_by_saved_path("floodshift_preprocessing_GLOBAL_LOCK_v1.json")
OPTICAL_NORM_PATH = locate_by_saved_path("optical_10band_train_normalization_exact.csv")
SAR_SCENE_STATS_PATH = locate_by_saved_path("train_raw_sar_scene_statistics.csv")

WARMSTART_PATH = locate_by_saved_path("FloodShift_FINAL_TerraMind_RIFT_best.pt")
REFERENCE_PROTOCOL_PATH = locate_by_saved_path("FloodShift_FINAL_TerraMind_RIFT_protocol.json", required=False)

with open(GLOBAL_LOCK_PATH, "r") as f:
    preprocessing_lock = json.load(f)
assert preprocessing_lock["status"] == "PERMANENTLY_FROZEN"
assert preprocessing_lock["preprocessing_development_closed"] is True

print("=" * 112)
print("P8-C02 — INPUTS FOUND")
print("=" * 112)
for p in [FINAL_MANIFEST_PATH, GLOBAL_LOCK_PATH, OPTICAL_NORM_PATH, SAR_SCENE_STATS_PATH, WARMSTART_PATH]:
    print(" ", p)
print("Reference protocol:", REFERENCE_PROTOCOL_PATH)
print("Lock              :", preprocessing_lock["status"])
print("P8-C02 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P8-C03 — MANIFEST + FROZEN TRAIN-ONLY NORMALIZATION
# ============================================================

manifest = pd.read_csv(FINAL_MANIFEST_PATH)

assert len(manifest) == 440
assert manifest["scene_id"].nunique() == 440

train_df = (
    manifest[manifest["split"] == "train"]
    .copy()
    .reset_index(drop=True)
)

val_df = (
    manifest[manifest["split"] == "validation"]
    .copy()
    .reset_index(drop=True)
)

test_df = (
    manifest[manifest["split"] == "test"]
    .copy()
    .reset_index(drop=True)
)

assert len(train_df) == 314
assert len(val_df) == 68
assert len(test_df) == 58

optical_norm = pd.read_csv(OPTICAL_NORM_PATH)

OPTICAL_BANDS = optical_norm["band_name"].tolist()

EXPECTED_OPTICAL_BANDS = [
    "B2", "B3", "B4", "B5", "B6",
    "B7", "B8", "B8A", "B11", "B12"
]

assert OPTICAL_BANDS == EXPECTED_OPTICAL_BANDS

OPTICAL_MEAN = optical_norm["mean"].to_numpy(np.float32)
OPTICAL_STD = optical_norm["std"].to_numpy(np.float32)

# Original S2 source order:
# B1 B2 B3 B4 B5 B6 B7 B8 B8A B9 B10 B11 B12
OPTICAL_SOURCE_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

sar_stats = pd.read_csv(SAR_SCENE_STATS_PATH)
assert len(sar_stats) == 314

n_all = sar_stats["valid_pixels"].to_numpy(np.float64)

def pooled_mean_std(mean_col, std_col):
    mu_all = sar_stats[mean_col].to_numpy(np.float64)
    sd_all = sar_stats[std_col].to_numpy(np.float64)

    valid = (
        np.isfinite(mu_all)
        & np.isfinite(sd_all)
        & (n_all > 0)
    )

    n = n_all[valid]
    mu = mu_all[valid]
    sd = sd_all[valid]

    total_n = n.sum()
    mean = np.sum(n * mu) / total_n
    second = np.sum(n * (sd**2 + mu**2)) / total_n
    var = max(second - mean**2, 0.0)

    return float(mean), float(np.sqrt(var))

VV_MEAN, VV_STD = pooled_mean_std(
    "vv_mean_db",
    "vv_std_db",
)

VH_MEAN, VH_STD = pooled_mean_std(
    "vh_mean_db",
    "vh_std_db",
)

SAR_MEAN = np.array(
    [VV_MEAN, VH_MEAN],
    dtype=np.float32,
)

SAR_STD = np.array(
    [VV_STD, VH_STD],
    dtype=np.float32,
)

assert np.isfinite(SAR_MEAN).all()
assert np.isfinite(SAR_STD).all()
assert (SAR_STD > 0).all()

print("=" * 112)
print("P8-C03 — DATA CONFIG")
print("=" * 112)
print("Train / Val / Test:", len(train_df), len(val_df), len(test_df))
print("Optical bands      :", OPTICAL_BANDS)
print("SAR mean [VV,VH]   :", SAR_MEAN)
print("SAR std  [VV,VH]   :", SAR_STD)
print("Reliability        : frozen continuous P(clear)")
print("Primary SAR input  : raw VV/VH")


In [ ]:
# ============================================================
# CELL: P8-C04 — TARGET + DATASET
#
# IMPORTANT:
# This intentionally preserves the CURRENT project's target mapping.
# It does NOT silently change label semantics mid-experiment.
# ============================================================

IGNORE_INDEX = 255
NUM_CLASSES = 3

CLASS_NAMES = [
    "non_flood",
    "permanent_water",
    "temporary_flood",
]

TILE_SIZE = 256
TILES_PER_SCENE = 4

TILE_OFFSETS = [
    (0, 0),
    (0, 256),
    (256, 0),
    (256, 256),
]


def resolve_scene_files(row):
    return {
        "sar": locate_by_saved_path(row["sar_raw_path"]),
        "optical": locate_by_saved_path(row["optical_raw_path"]),
        "flood": locate_by_saved_path(row["flood_label_path"]),
        "jrc": locate_by_saved_path(row["jrc_water_path"]),
        "reliability": locate_by_saved_path(
            row["optical_reliability_path"]
        ),
    }


def build_three_class_target(flood, jrc):
    flood = np.asarray(flood)
    jrc = np.asarray(jrc)

    target = np.full(
        flood.shape,
        IGNORE_INDEX,
        dtype=np.uint8,
    )

    valid = flood != -1

    target[valid] = 0

    # Current frozen project semantics:
    # any valid JRC permanent-water pixel -> class 1
    target[
        valid & (jrc == 1)
    ] = 1

    # flood-positive but not JRC permanent water -> class 2
    target[
        valid
        & (jrc != 1)
        & (flood == 1)
    ] = 2

    return target


def normalize_sar(sar):
    sar = sar.astype(
        np.float32,
        copy=False,
    )

    valid = np.isfinite(sar).all(axis=0)

    out = np.zeros_like(
        sar,
        dtype=np.float32,
    )

    for c in range(2):
        m = valid & np.isfinite(sar[c])

        out[c, m] = (
            sar[c, m] - SAR_MEAN[c]
        ) / SAR_STD[c]

    return out


def normalize_optical(optical_13):
    optical_13 = optical_13.astype(
        np.float32,
        copy=False,
    )

    valid = np.all(
        np.isfinite(optical_13)
        & (optical_13 != 0),
        axis=0,
    )

    reflectance = optical_13 / 10000.0

    optical_10 = reflectance[
        OPTICAL_SOURCE_INDICES
    ]

    out = (
        optical_10
        - OPTICAL_MEAN[:, None, None]
    ) / OPTICAL_STD[:, None, None]

    out = out.astype(
        np.float32,
        copy=False,
    )

    # Frozen Notebook-03 convention.
    out[:, ~valid] = 0.0

    return out


def augment_all(
    sar,
    optical,
    reliability,
    target,
):
    # D4-aligned augmentation only.
    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=2)
        optical = np.flip(optical, axis=2)
        reliability = np.flip(
            reliability,
            axis=1,
        )
        target = np.flip(
            target,
            axis=1,
        )

    if np.random.rand() < 0.5:
        sar = np.flip(sar, axis=1)
        optical = np.flip(optical, axis=1)
        reliability = np.flip(
            reliability,
            axis=0,
        )
        target = np.flip(
            target,
            axis=0,
        )

    k = np.random.randint(0, 4)

    if k:
        sar = np.rot90(
            sar,
            k=k,
            axes=(1, 2),
        )

        optical = np.rot90(
            optical,
            k=k,
            axes=(1, 2),
        )

        reliability = np.rot90(
            reliability,
            k=k,
            axes=(0, 1),
        )

        target = np.rot90(
            target,
            k=k,
            axes=(0, 1),
        )

    return (
        sar.copy(),
        optical.copy(),
        reliability.copy(),
        target.copy(),
    )


class FixedEvalDataset(Dataset):
    def __init__(
        self,
        dataframe,
        train_mode=False,
    ):
        self.df = dataframe.reset_index(
            drop=True
        )

        self.train_mode = train_mode

    def __len__(self):
        return (
            len(self.df)
            * TILES_PER_SCENE
        )

    def __getitem__(self, index):
        scene_index = (
            index // TILES_PER_SCENE
        )

        tile_index = (
            index % TILES_PER_SCENE
        )

        row = self.df.iloc[
            scene_index
        ]

        scene_id = str(
            row["scene_id"]
        )

        y0, x0 = TILE_OFFSETS[
            tile_index
        ]

        window = Window(
            x0,
            y0,
            TILE_SIZE,
            TILE_SIZE,
        )

        paths = resolve_scene_files(
            row
        )

        with rasterio.open(
            paths["sar"]
        ) as src:
            sar = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["optical"]
        ) as src:
            optical_13 = src.read(
                window=window
            ).astype(np.float32)

        with rasterio.open(
            paths["flood"]
        ) as src:
            flood = src.read(
                1,
                window=window,
            )

        with rasterio.open(
            paths["jrc"]
        ) as src:
            jrc = src.read(
                1,
                window=window,
            )

        reliability_full = np.load(
            paths["reliability"],
            mmap_mode="r",
        )

        reliability = np.asarray(
            reliability_full[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE,
            ],
            dtype=np.float32,
        )

        sar = normalize_sar(sar)

        optical = normalize_optical(
            optical_13
        )

        target = build_three_class_target(
            flood,
            jrc,
        )

        reliability = np.clip(
            reliability,
            0.0,
            1.0,
        )

        if self.train_mode:
            (
                sar,
                optical,
                reliability,
                target,
            ) = augment_all(
                sar,
                optical,
                reliability,
                target,
            )

        return {
            "sar": torch.from_numpy(
                sar
            ),
            "optical": torch.from_numpy(
                optical
            ),
            "reliability": torch.from_numpy(
                reliability[
                    None,
                    ...,
                ]
            ),
            "target": torch.from_numpy(
                target.astype(
                    np.int64
                )
            ),
            "scene_id": scene_id,
            "tile_index": tile_index,
        }


In [ ]:
# ============================================================
# CELL: P8-C05 — TARGET AUDIT + CLASS-AWARE CROP CANDIDATES
# ============================================================
# Same frozen target mapping as 04E. We additionally build a bounded
# candidate bank for training-only stochastic crop oversampling.

class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
ignored_count = 0
MAX_CANDIDATES_PER_KIND = 4096
candidate_bank = {}
scene_weights = []

rng_global = np.random.default_rng(SEED)

def cap_indices(mask, rng, cap=MAX_CANDIDATES_PER_KIND):
    flat = np.flatnonzero(mask.reshape(-1))
    if flat.size > cap:
        flat = rng.choice(flat, size=cap, replace=False)
    return flat.astype(np.int32, copy=False)

for position, (_, row) in enumerate(train_df.iterrows(), start=1):
    paths = resolve_scene_files(row)
    with rasterio.open(paths["flood"]) as src: flood = src.read(1)
    with rasterio.open(paths["jrc"]) as src: jrc = src.read(1)
    target = build_three_class_target(flood, jrc)

    for c in range(NUM_CLASSES): class_counts[c] += int((target == c).sum())
    ignored_count += int((target == IGNORE_INDEX).sum())

    scene_id = str(row["scene_id"])
    local_rng = np.random.default_rng(SEED + position * 1009)
    temp = target == 2
    perm = target == 1
    valid = target != IGNORE_INDEX

    # Difficult class-1/class-2 neighborhood. If empty, water edges are used.
    near_perm = ndimage.binary_dilation(perm, iterations=5)
    near_temp = ndimage.binary_dilation(temp, iterations=5)
    mixed = near_perm & near_temp & valid
    if not mixed.any():
        water = perm | temp
        mixed = (ndimage.binary_dilation(water, iterations=3) ^ ndimage.binary_erosion(water, iterations=3)) & valid

    candidate_bank[scene_id] = {
        "temp": cap_indices(temp, local_rng),
        "perm": cap_indices(perm, local_rng),
        "mixed": cap_indices(mixed, local_rng),
    }

    has_temp = candidate_bank[scene_id]["temp"].size > 0
    has_perm = candidate_bank[scene_id]["perm"].size > 0
    has_mixed = candidate_bank[scene_id]["mixed"].size > 0
    # Fixed oversampling at scene level; no validation/test information is used.
    scene_weights.append(1.0 + 1.25*has_temp + 0.45*has_perm + 0.35*has_mixed)

    if position == 1 or position % 75 == 0 or position == len(train_df):
        print(f"[{position:03d}/{len(train_df)}] candidate audit")

valid_total = class_counts.sum()
frequencies = class_counts / valid_total
weights_np = 1.0 / np.sqrt(np.maximum(frequencies, 1e-12))
weights_np = weights_np / weights_np.mean()
weights_np = np.clip(weights_np, 0.25, 2.00)
weights_np = weights_np / weights_np.mean()
CLASS_WEIGHTS = torch.tensor(weights_np, dtype=torch.float32, device=DEVICE)

print("\n" + "=" * 112)
print("P8-C05 — TRAIN DISTRIBUTION + CROP BANK")
print("=" * 112)
for c, name in enumerate(CLASS_NAMES):
    print(f"{c} {name:18s}: {class_counts[c]:,} ({frequencies[c]*100:.2f}%) weight={weights_np[c]:.4f}")
print("Ignored:", f"{ignored_count:,}")
print("Candidate scenes — temp:", sum(v["temp"].size>0 for v in candidate_bank.values()),
      "perm:", sum(v["perm"].size>0 for v in candidate_bank.values()),
      "mixed:", sum(v["mixed"].size>0 for v in candidate_bank.values()))
print("P8-C05 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P8-C06 — CLASS-AWARE TRAIN DATASET + FIXED EVAL LOADERS
# ============================================================
CROPS_PER_SCENE = 6
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
NUM_WORKERS = 2

# Fixed, predeclared training crop probabilities.
CROP_MODE_PROBS = {
    "temp": 0.50,
    "perm": 0.25,
    "mixed": 0.15,
    "uniform": 0.10,
}

class ClassAwareCropDataset(Dataset):
    def __init__(self, dataframe, candidate_bank, train_mode=True):
        self.df = dataframe.reset_index(drop=True)
        self.candidate_bank = candidate_bank
        self.train_mode = train_mode

    def __len__(self):
        return len(self.df) * CROPS_PER_SCENE

    def _choose_origin(self, row, scene_id):
        mode = np.random.choice(list(CROP_MODE_PROBS.keys()), p=list(CROP_MODE_PROBS.values()))
        bank = self.candidate_bank[scene_id]
        H = W = 512
        if mode != "uniform" and bank[mode].size > 0:
            flat = int(np.random.choice(bank[mode]))
            cy, cx = divmod(flat, W)
            cy += np.random.randint(-48, 49)
            cx += np.random.randint(-48, 49)
            y0 = int(np.clip(cy - TILE_SIZE//2, 0, H - TILE_SIZE))
            x0 = int(np.clip(cx - TILE_SIZE//2, 0, W - TILE_SIZE))
        else:
            y0 = int(np.random.randint(0, H - TILE_SIZE + 1))
            x0 = int(np.random.randint(0, W - TILE_SIZE + 1))
        return y0, x0, mode

    def __getitem__(self, index):
        scene_index = index // CROPS_PER_SCENE
        row = self.df.iloc[scene_index]
        scene_id = str(row["scene_id"])
        y0, x0, mode = self._choose_origin(row, scene_id)
        window = Window(x0, y0, TILE_SIZE, TILE_SIZE)
        paths = resolve_scene_files(row)

        with rasterio.open(paths["sar"]) as src: sar = src.read(window=window).astype(np.float32)
        with rasterio.open(paths["optical"]) as src: optical_13 = src.read(window=window).astype(np.float32)
        with rasterio.open(paths["flood"]) as src: flood = src.read(1, window=window)
        with rasterio.open(paths["jrc"]) as src: jrc = src.read(1, window=window)
        rel_full = np.load(paths["reliability"], mmap_mode="r")
        reliability = np.asarray(rel_full[y0:y0+TILE_SIZE, x0:x0+TILE_SIZE], dtype=np.float32)

        sar = normalize_sar(sar)
        optical = normalize_optical(optical_13)
        target = build_three_class_target(flood, jrc)
        reliability = np.clip(reliability, 0.0, 1.0)

        if self.train_mode:
            sar, optical, reliability, target = augment_all(sar, optical, reliability, target)

        return {
            "sar": torch.from_numpy(sar),
            "optical": torch.from_numpy(optical),
            "reliability": torch.from_numpy(reliability[None, ...]),
            "target": torch.from_numpy(target.astype(np.int64)),
            "scene_id": scene_id,
            "crop_mode": mode,
        }


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed); random.seed(worker_seed)

train_dataset = ClassAwareCropDataset(train_df, candidate_bank, train_mode=True)
val_dataset = FixedEvalDataset(val_df, train_mode=False)
test_dataset = FixedEvalDataset(test_df, train_mode=False)

# Scene-level fixed oversampling. Every virtual crop from a scene receives its scene weight.
item_weights = []
for i in range(len(train_dataset)):
    scene_idx = i // CROPS_PER_SCENE
    item_weights.append(scene_weights[scene_idx])
item_weights = torch.tensor(item_weights, dtype=torch.double)


def make_loaders(seed=SEED):
    g = torch.Generator().manual_seed(seed)
    sampler = WeightedRandomSampler(item_weights, num_samples=len(item_weights), replacement=True, generator=g)
    common = dict(num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available(), worker_init_fn=seed_worker,
                  persistent_workers=(NUM_WORKERS > 0))
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, drop_last=False, **common)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **common)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, **common)
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = make_loaders()
batch = next(iter(train_loader))
print("=" * 112)
print("P8-C06 — BATCH CHECK")
print("=" * 112)
print("Train samples/epoch:", len(train_dataset))
print("SAR        :", tuple(batch["sar"].shape))
print("Optical    :", tuple(batch["optical"].shape))
print("Reliability:", tuple(batch["reliability"].shape))
print("Target     :", tuple(batch["target"].shape))
print("Crop modes :", batch["crop_mode"])
assert batch["sar"].shape[1:] == (2,256,256)
assert batch["optical"].shape[1:] == (10,256,256)
assert torch.isfinite(batch["sar"]).all() and torch.isfinite(batch["optical"]).all()
print("P8-C06 COMPLETE ✅")


## 04F architecture strategy

The exact 04E TerraMind-RIFT model is reconstructed with **the same state-dict names** and loaded strictly from the sanitized final checkpoint. `pretrained=False` is intentional: the 04E checkpoint already contains the complete TerraMind weights and the interpolated 256×256 positional embeddings, so re-downloading the 1.52 GB foundation checkpoint is unnecessary.

04F then wraps it with a tiny zero-initialized hierarchical residual refiner. Because its last layer starts at zero, the first forward pass is numerically identical to 04E; every change must be learned from the train split.


In [ ]:
# ============================================================
# CELL: P8-C07 — FINAL TERRAMIND-RIFT ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# TerraMind pretrained input conventions
# Official Sen1Floods11 config statistics.
# ------------------------------------------------------------

TM_S2_ALL_MEAN = np.array(
    [
        2357.089,
        2137.385,
        2018.788,
        2082.986,
        2295.651,
        2854.537,
        3122.849,
        3040.560,
        3306.481,
        1473.847,
        506.070,
        2472.825,
        1838.929,
    ],
    dtype=np.float32,
)

TM_S2_ALL_STD = np.array(
    [
        1624.683,
        1675.806,
        1557.708,
        1833.702,
        1823.738,
        1733.977,
        1732.131,
        1679.732,
        1727.260,
        1024.687,
        442.165,
        1331.411,
        1160.419,
    ],
    dtype=np.float32,
)

TM_S2_SUBSET_INDICES = np.array(
    [1, 2, 3, 4, 5, 6, 7, 8, 11, 12],
    dtype=np.int64,
)

TM_S2_MEAN = (
    TM_S2_ALL_MEAN[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S2_STD = (
    TM_S2_ALL_STD[
        TM_S2_SUBSET_INDICES
    ]
)

TM_S1_MEAN = np.array(
    [-12.599, -20.293],
    dtype=np.float32,
)

TM_S1_STD = np.array(
    [5.195, 5.890],
    dtype=np.float32,
)

TM_OPTICAL_BAND_NAMES = [
    "BLUE",
    "GREEN",
    "RED",
    "RED_EDGE_1",
    "RED_EDGE_2",
    "RED_EDGE_3",
    "NIR_BROAD",
    "NIR_NARROW",
    "SWIR_1",
    "SWIR_2",
]

TM_LAYER_INDICES = [
    2,
    5,
    8,
    11,
]

TM_EMBED_DIM = 768
PYRAMID_CHANNELS = [
    64,
    128,
    256,
    512,
]

PYRAMID_SIZES = [
    128,
    64,
    32,
    16,
]


def make_group_norm(channels):
    for groups in [
        32,
        16,
        8,
        4,
        2,
        1,
    ]:
        if (
            channels
            % groups
            == 0
        ):
            return nn.GroupNorm(
                groups,
                channels,
            )

    raise RuntimeError(
        "No GroupNorm divisor."
    )


class ConvNormAct(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
    ):
        super().__init__()

        padding = (
            kernel_size // 2
        )

        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                padding=padding,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SEBlock(nn.Module):
    def __init__(
        self,
        channels,
        reduction=8,
    ):
        super().__init__()

        hidden = max(
            channels // reduction,
            8,
        )

        self.fc1 = nn.Conv2d(
            channels,
            hidden,
            kernel_size=1,
        )

        self.fc2 = nn.Conv2d(
            hidden,
            channels,
            kernel_size=1,
        )

    def forward(self, x):
        w = F.adaptive_avg_pool2d(
            x,
            1,
        )

        w = F.gelu(
            self.fc1(w)
        )

        w = torch.sigmoid(
            self.fc2(w)
        )

        return x * w


class ConvNeXtRefine(nn.Module):
    def __init__(
        self,
        channels,
        drop=0.0,
    ):
        super().__init__()

        self.dw = nn.Conv2d(
            channels,
            channels,
            kernel_size=7,
            padding=3,
            groups=channels,
        )

        self.norm = make_group_norm(
            channels
        )

        self.pw1 = nn.Conv2d(
            channels,
            channels * 4,
            kernel_size=1,
        )

        self.pw2 = nn.Conv2d(
            channels * 4,
            channels,
            kernel_size=1,
        )

        self.drop = nn.Dropout2d(
            drop
        )

        self.se = SEBlock(
            channels
        )

    def forward(self, x):
        residual = x

        x = self.dw(x)

        x = self.norm(x)

        x = F.gelu(
            self.pw1(x)
        )

        x = self.drop(
            self.pw2(x)
        )

        x = self.se(x)

        return residual + x


class ModalityProjector(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        out_size,
    ):
        super().__init__()

        self.out_size = out_size

        self.proj = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                bias=False,
            ),
            make_group_norm(
                out_channels
            ),
            nn.GELU(),
            ConvNeXtRefine(
                out_channels,
                drop=0.0,
            ),
        )

    def forward(self, x):
        x = self.proj(x)

        if (
            x.shape[-2:]
            != (
                self.out_size,
                self.out_size,
            )
        ):
            x = F.interpolate(
                x,
                size=(
                    self.out_size,
                    self.out_size,
                ),
                mode="bilinear",
                align_corners=False,
            )

        return x


class ReliabilityRouter(nn.Module):
    """
    Spatial SAR-optical router.

    P(clear) is used as a prior on the optical route,
    while the learned router can still override that prior.
    """

    def __init__(
        self,
        channels,
        hidden=64,
        prior_strength=0.50,
    ):
        super().__init__()

        hidden = min(
            hidden,
            max(
                channels // 2,
                16,
            ),
        )

        self.prior_strength = (
            float(
                prior_strength
            )
        )

        self.net = nn.Sequential(
            nn.Conv2d(
                channels * 2 + 1,
                hidden,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(
                hidden
            ),
            nn.GELU(),
            nn.Conv2d(
                hidden,
                2,
                kernel_size=1,
            ),
        )

        # Start near neutral learned routing.
        nn.init.zeros_(
            self.net[-1].weight
        )

        nn.init.zeros_(
            self.net[-1].bias
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        r = F.interpolate(
            pclear,
            size=sar.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        r = r.clamp(
            0.0,
            1.0,
        )

        logits = self.net(
            torch.cat(
                [
                    sar,
                    optical,
                    r,
                ],
                dim=1,
            )
        )

        # Optical prior. Clamp avoids -inf at P(clear)=0.
        logits[
            :,
            1:2,
        ] = (
            logits[
                :,
                1:2,
            ]
            + self.prior_strength
            * torch.log(
                r.clamp(
                    min=0.05,
                    max=1.0,
                )
            )
        )

        weights = torch.softmax(
            logits,
            dim=1,
        )

        w_sar = weights[
            :,
            0:1,
        ]

        w_opt = weights[
            :,
            1:2,
        ]

        fused = (
            w_sar * sar
            + w_opt * optical
        )

        return (
            fused,
            weights,
        )


class DeepRIFT(nn.Module):
    """
    Zero-initialized reliability-conditioned cross attention.

    When P(clear) is high:
      SAR can borrow more optical context.

    When P(clear) is low:
      optical can borrow more SAR context.

    Output projections start at zero, so the block is an
    exact identity at initialization.
    """

    def __init__(
        self,
        channels=512,
        heads=8,
        dropout=0.0,
    ):
        super().__init__()

        self.channels = channels

        self.sar_norm = nn.LayerNorm(
            channels
        )

        self.opt_norm = nn.LayerNorm(
            channels
        )

        self.sar_from_opt = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.opt_from_sar = (
            nn.MultiheadAttention(
                channels,
                num_heads=heads,
                dropout=dropout,
                batch_first=True,
            )
        )

        self.sar_out = nn.Linear(
            channels,
            channels,
        )

        self.opt_out = nn.Linear(
            channels,
            channels,
        )

        nn.init.zeros_(
            self.sar_out.weight
        )

        nn.init.zeros_(
            self.sar_out.bias
        )

        nn.init.zeros_(
            self.opt_out.weight
        )

        nn.init.zeros_(
            self.opt_out.bias
        )

    @staticmethod
    def _flatten(x):
        return (
            x.flatten(2)
            .transpose(1, 2)
            .contiguous()
        )

    @staticmethod
    def _unflatten(
        tokens,
        height,
        width,
    ):
        return (
            tokens
            .transpose(1, 2)
            .reshape(
                tokens.shape[0],
                tokens.shape[2],
                height,
                width,
            )
            .contiguous()
        )

    def forward(
        self,
        sar,
        optical,
        pclear,
    ):
        b, c, h, w = sar.shape

        sar_t = self._flatten(
            sar
        )

        opt_t = self._flatten(
            optical
        )

        r = F.adaptive_avg_pool2d(
            pclear,
            output_size=(h, w),
        )

        r_t = (
            r.flatten(2)
            .transpose(1, 2)
            .clamp(0.0, 1.0)
        )

        sar_n = self.sar_norm(
            sar_t
        )

        opt_n = self.opt_norm(
            opt_t
        )

        # Clear optical context receives larger K/V magnitude.
        opt_context = (
            opt_n
            * (
                0.25
                + 0.75 * r_t
            )
        )

        sar_delta, _ = (
            self.sar_from_opt(
                query=sar_n,
                key=opt_context,
                value=opt_context,
                need_weights=False,
            )
        )

        # Under low P(clear), the optical branch borrows SAR more.
        opt_delta, _ = (
            self.opt_from_sar(
                query=opt_n,
                key=sar_n,
                value=sar_n,
                need_weights=False,
            )
        )

        sar_t = (
            sar_t
            + r_t
            * self.sar_out(
                sar_delta
            )
        )

        opt_t = (
            opt_t
            + (1.0 - r_t)
            * self.opt_out(
                opt_delta
            )
        )

        sar = self._unflatten(
            sar_t,
            h,
            w,
        )

        optical = self._unflatten(
            opt_t,
            h,
            w,
        )

        return sar, optical


class DecoderBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
    ):
        super().__init__()

        self.fuse = nn.Sequential(
            ConvNormAct(
                in_channels
                + skip_channels,
                out_channels,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
            ConvNeXtRefine(
                out_channels,
                drop=0.05,
            ),
        )

    def forward(
        self,
        x,
        skip,
    ):
        x = F.interpolate(
            x,
            size=skip.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )

        x = torch.cat(
            [
                x,
                skip,
            ],
            dim=1,
        )

        return self.fuse(x)


class FloodShiftTerraMindRIFT(nn.Module):
    def __init__(
        self,
        num_classes=3,
        pretrained_backbone=False,
    ):
        super().__init__()

        # concat is used deliberately:
        # each spatial token contains S2 and S1 embeddings
        # concatenated in the modality order supplied below.
        self.backbone = (
            BACKBONE_REGISTRY.build(
                "terramind_v1_base",
                pretrained=pretrained_backbone,
                img_size=TILE_SIZE,
                modalities=[
                    "S2L1C",
                    "S1GRD",
                ],
                bands={
                    "S2L1C":
                        TM_OPTICAL_BAND_NAMES,
                },
                merge_method="concat",
            )
        )

        self.register_buffer(
            "project_opt_mean",
            torch.tensor(
                OPTICAL_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_opt_std",
            torch.tensor(
                OPTICAL_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_mean",
            torch.tensor(
                SAR_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "project_sar_std",
            torch.tensor(
                SAR_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_mean",
            torch.tensor(
                TM_S2_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_opt_std",
            torch.tensor(
                TM_S2_STD,
                dtype=torch.float32,
            ).view(
                1,
                10,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_mean",
            torch.tensor(
                TM_S1_MEAN,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.register_buffer(
            "tm_sar_std",
            torch.tensor(
                TM_S1_STD,
                dtype=torch.float32,
            ).view(
                1,
                2,
                1,
                1,
            ),
            persistent=True,
        )

        self.sar_projectors = (
            nn.ModuleList()
        )

        self.opt_projectors = (
            nn.ModuleList()
        )

        self.routers = nn.ModuleList()

        for (
            channels,
            out_size,
        ) in zip(
            PYRAMID_CHANNELS,
            PYRAMID_SIZES,
        ):
            self.sar_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.opt_projectors.append(
                ModalityProjector(
                    TM_EMBED_DIM,
                    channels,
                    out_size,
                )
            )

            self.routers.append(
                ReliabilityRouter(
                    channels,
                    prior_strength=0.50,
                )
            )

        self.deep_rift = DeepRIFT(
            channels=PYRAMID_CHANNELS[-1],
            heads=8,
            dropout=0.0,
        )

        self.deep_context = nn.Sequential(
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
            ConvNeXtRefine(
                PYRAMID_CHANNELS[-1],
                drop=0.05,
            ),
        )

        self.decoder3 = DecoderBlock(
            512,
            256,
            256,
        )

        self.decoder2 = DecoderBlock(
            256,
            128,
            128,
        )

        self.decoder1 = DecoderBlock(
            128,
            64,
            64,
        )

        self.final_refine = nn.Sequential(
            ConvNormAct(
                64,
                64,
                kernel_size=3,
            ),
            ConvNeXtRefine(
                64,
                drop=0.05,
            ),
        )

        self.segmentation_head = (
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            )
        )

        self.aux_head = nn.Sequential(
            nn.Conv2d(
                128,
                64,
                kernel_size=3,
                padding=1,
                bias=False,
            ),
            make_group_norm(64),
            nn.GELU(),
            nn.Conv2d(
                64,
                num_classes,
                kernel_size=1,
            ),
        )

    def to_terramind_inputs(
        self,
        sar_z,
        optical_z,
    ):
        # Detect frozen invalid-fill pixels BEFORE inverse affine.
        opt_invalid = (
            optical_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        sar_invalid = (
            sar_z
            .abs()
            .sum(
                dim=1,
                keepdim=True,
            )
            < 1e-12
        )

        # Frozen optical z-score -> reflectance -> DN.
        optical_reflectance = (
            optical_z
            * self.project_opt_std
            + self.project_opt_mean
        )

        optical_dn = (
            optical_reflectance
            * 10000.0
        )

        optical_tm = (
            optical_dn
            - self.tm_opt_mean
        ) / self.tm_opt_std

        # Frozen SAR z-score -> raw dB.
        sar_db = (
            sar_z
            * self.project_sar_std
            + self.project_sar_mean
        )

        sar_tm = (
            sar_db
            - self.tm_sar_mean
        ) / self.tm_sar_std

        # Preserve invalid-fill convention in model space.
        optical_tm = optical_tm.masked_fill(
            opt_invalid,
            0.0,
        )

        sar_tm = sar_tm.masked_fill(
            sar_invalid,
            0.0,
        )

        return (
            sar_tm,
            optical_tm,
        )

    @staticmethod
    def tokens_to_map(tokens):
        b, n, c = tokens.shape

        side = int(
            math.isqrt(n)
        )

        if (
            side * side
            != n
        ):
            raise RuntimeError(
                f"TerraMind token count {n} "
                "is not a square grid."
            )

        return (
            tokens
            .transpose(1, 2)
            .reshape(
                b,
                c,
                side,
                side,
            )
            .contiguous()
        )

    def extract_modality_tokens(
        self,
        tm_outputs,
    ):
        if (
            not isinstance(
                tm_outputs,
                (list, tuple),
            )
            or len(tm_outputs) < 12
        ):
            raise RuntimeError(
                "Unexpected TerraMind output. "
                "Expected list of 12 transformer-layer outputs."
            )

        sar_tokens = []
        opt_tokens = []

        for layer_index in TM_LAYER_INDICES:
            tokens = tm_outputs[
                layer_index
            ]

            if (
                not torch.is_tensor(tokens)
                or tokens.ndim != 3
            ):
                raise RuntimeError(
                    "TerraMind concat output should be "
                    "[B, tokens, embedding]."
                )

            expected_dim = (
                TM_EMBED_DIM * 2
            )

            if (
                tokens.shape[-1]
                != expected_dim
            ):
                raise RuntimeError(
                    "Expected concat embedding dimension "
                    f"{expected_dim}, got "
                    f"{tokens.shape[-1]}. "
                    "Check TerraTorch version and merge_method."
                )

            # modalities were supplied in order:
            # [S2L1C, S1GRD]
            optical_part = tokens[
                :,
                :,
                :TM_EMBED_DIM,
            ]

            sar_part = tokens[
                :,
                :,
                TM_EMBED_DIM:,
            ]

            opt_tokens.append(
                optical_part
            )

            sar_tokens.append(
                sar_part
            )

        return (
            sar_tokens,
            opt_tokens,
        )

    def forward(
        self,
        sar,
        optical,
        reliability,
        return_aux=False,
    ):
        (
            sar_tm,
            optical_tm,
        ) = self.to_terramind_inputs(
            sar,
            optical,
        )

        tm_outputs = self.backbone(
            {
                "S2L1C": optical_tm,
                "S1GRD": sar_tm,
            }
        )

        (
            sar_tokens,
            opt_tokens,
        ) = self.extract_modality_tokens(
            tm_outputs
        )

        sar_maps = []
        opt_maps = []

        for i in range(4):
            sar_map = (
                self.tokens_to_map(
                    sar_tokens[i]
                )
            )

            opt_map = (
                self.tokens_to_map(
                    opt_tokens[i]
                )
            )

            sar_map = (
                self.sar_projectors[i](
                    sar_map
                )
            )

            opt_map = (
                self.opt_projectors[i](
                    opt_map
                )
            )

            sar_maps.append(
                sar_map
            )

            opt_maps.append(
                opt_map
            )

        # RIFT only at deepest 16x16 level.
        (
            sar_maps[-1],
            opt_maps[-1],
        ) = self.deep_rift(
            sar_maps[-1],
            opt_maps[-1],
            reliability,
        )

        fused = []
        router_weights = []

        for i in range(4):
            (
                fused_i,
                weights_i,
            ) = self.routers[i](
                sar_maps[i],
                opt_maps[i],
                reliability,
            )

            fused.append(
                fused_i
            )

            router_weights.append(
                weights_i
            )

        f0, f1, f2, f3 = fused

        x = self.deep_context(
            f3
        )

        x = self.decoder3(
            x,
            f2,
        )

        x = self.decoder2(
            x,
            f1,
        )

        aux_logits = self.aux_head(
            x
        )

        x = self.decoder1(
            x,
            f0,
        )

        x = F.interpolate(
            x,
            size=(
                TILE_SIZE,
                TILE_SIZE,
            ),
            mode="bilinear",
            align_corners=False,
        )

        x = self.final_refine(
            x
        )

        logits = (
            self.segmentation_head(
                x
            )
        )

        if return_aux:
            return {
                "logits": logits,
                "aux_logits": aux_logits,
                "router_weights": router_weights,
            }

        return logits


In [ ]:
# ============================================================
# CELL: P8-C08 — LOAD FROZEN 04E + ZERO-INIT HIERARCHICAL REFINER
# ============================================================
EXPECTED_04E_SHA256 = "6ab8ad8b41aa64b4ceee403539d8554a14f61cab235a0f0841ef13f77e390822"

# Hash audit
sha = hashlib.sha256()
with open(WARMSTART_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1024*1024), b""):
        sha.update(chunk)
warm_sha = sha.hexdigest()
print("04E checkpoint SHA256:", warm_sha)
if warm_sha != EXPECTED_04E_SHA256:
    print("⚠️ Hash differs from the archived 04E reference. Continuing only after state/metric audits below.")
else:
    print("✅ Exact archived 04E checkpoint detected")

base_model_04e = FloodShiftTerraMindRIFT(num_classes=NUM_CLASSES, pretrained_backbone=False)
payload = torch.load(WARMSTART_PATH, map_location="cpu", weights_only=False)
state = payload["state_dict"] if "state_dict" in payload else payload

# Defensive sanitation: only the known TerraMind missing-modality fallback may be non-finite.
key = "backbone.missing_mod_token"
if key in state and not torch.isfinite(state[key]).all():
    print("Sanitizing unused", key)
    state[key] = torch.zeros_like(state[key])

missing, unexpected = base_model_04e.load_state_dict(state, strict=False)
assert not missing, f"Missing checkpoint keys: {missing[:20]}"
assert not unexpected, f"Unexpected checkpoint keys: {unexpected[:20]}"

bad = [n for n,t in base_model_04e.state_dict().items() if torch.is_floating_point(t) and not torch.isfinite(t).all()]
assert not bad, f"Non-finite model tensors remain: {bad}"

class WaterTypeResidualRefiner(nn.Module):
    """Conservative hierarchical correction of an already strong 3-class predictor.

    Input: base probabilities + SAR + optical + P(clear).
    Outputs two residual fields:
      water_delta: background <-> {permanent,temp}
      type_delta : permanent <-> temporary
    Last conv is zero initialized, so delta == 0 at initialization.
    """
    def __init__(self, in_channels=16, hidden=48):
        super().__init__()
        self.net = nn.Sequential(
            ConvNormAct(in_channels, hidden, 3),
            ConvNeXtRefine(hidden, drop=0.05),
            ConvNormAct(hidden, hidden, 3),
            nn.Conv2d(hidden, 2, kernel_size=1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, base_logits, sar, optical, reliability):
        base_prob = torch.softmax(base_logits.float(), dim=1).to(base_logits.dtype)
        x = torch.cat([base_prob, sar, optical, reliability], dim=1)
        return self.net(x)

class FloodShift04F(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.refiner = WaterTypeResidualRefiner()

    @property
    def backbone(self):
        return self.base.backbone

    def forward(self, sar, optical, reliability, return_aux=False):
        base_out = self.base(sar, optical, reliability, return_aux=True)
        base_logits = base_out["logits"]
        correction = self.refiner(base_logits, sar, optical, reliability)
        water_delta = correction[:, 0:1]
        type_delta = correction[:, 1:2]

        # Hierarchical residual composition; exact identity when correction == 0.
        logits = base_logits.clone()
        logits[:, 0:1] = logits[:, 0:1] - 0.50 * water_delta
        logits[:, 1:2] = logits[:, 1:2] + 0.50 * water_delta - 0.50 * type_delta
        logits[:, 2:3] = logits[:, 2:3] + 0.50 * water_delta + 0.50 * type_delta

        if return_aux:
            return {
                "logits": logits,
                "base_logits": base_logits,
                "aux_logits": base_out["aux_logits"],
                "router_weights": base_out["router_weights"],
                "correction": correction,
            }
        return logits

model = FloodShift04F(base_model_04e).to(DEVICE)

# At initialization, refiner must reproduce 04E exactly.
model.eval()
with torch.inference_mode():
    tiny_sar = batch["sar"][:1].to(DEVICE)
    tiny_opt = batch["optical"][:1].to(DEVICE)
    tiny_rel = batch["reliability"][:1].to(DEVICE)
    out04f = model(tiny_sar, tiny_opt, tiny_rel, return_aux=True)
    max_identity_diff = (out04f["logits"] - out04f["base_logits"]).abs().max().item()

print("Max 04F-vs-04E init logit diff:", f"{max_identity_diff:.3e}")
assert max_identity_diff == 0.0
assert torch.isfinite(out04f["logits"]).all()
print("Main logits:", tuple(out04f["logits"].shape))
print("04F total params:", f"{sum(p.numel() for p in model.parameters())/1e6:.2f} M")
print("P8-C08 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P8-C09 — TARGETED 3-CLASS LOSS
# ============================================================
# Sum = 1.00. Every component is computed in FP32 for stability.
W_CE       = 0.28
W_LOVASZ   = 0.20
W_WTYPE    = 0.18
W_TEMP_TV  = 0.10
W_PERM_TV  = 0.08
W_WATER    = 0.07
W_BOUNDARY = 0.06
W_AUX      = 0.03


def lovasz_grad(gt_sorted):
    p = len(gt_sorted); gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union = gts + (1.0 - gt_sorted).float().cumsum(0)
    j = 1.0 - intersection / union.clamp_min(1e-7)
    if p > 1: j[1:p] = j[1:p] - j[:-1]
    return j

def lovasz_softmax_loss(logits, target):
    probs = torch.softmax(logits.float(), dim=1)
    valid = target != IGNORE_INDEX
    if valid.sum() == 0: return logits.sum() * 0.0
    p = probs.permute(0,2,3,1)[valid]
    y = target[valid]
    losses=[]
    for c in range(NUM_CLASSES):
        fg=(y==c).float()
        if fg.sum()==0: continue
        err=(fg-p[:,c]).abs()
        err_s, perm=torch.sort(err, descending=True)
        losses.append(torch.dot(err_s, lovasz_grad(fg[perm])))
    return torch.stack(losses).mean() if losses else logits.sum()*0.0

def focal_tversky_class(logits, target, cls, alpha=0.60, beta=0.40, gamma=0.75):
    valid = target != IGNORE_INDEX
    if valid.sum()==0: return logits.sum()*0.0
    p=torch.softmax(logits.float(),dim=1)[:,cls][valid]
    g=(target==cls).float()[valid]
    if g.sum()==0: return p.sum()*0.0
    tp=(p*g).sum(); fp=(p*(1-g)).sum(); fn=((1-p)*g).sum()
    tv=(tp+1.0)/(tp+alpha*fp+beta*fn+1.0)
    return (1-tv).pow(gamma)

def conditional_water_type_ce(logits, target):
    water = (target==1)|(target==2)
    if water.sum()==0: return logits.sum()*0.0
    two = logits[:,1:3].float().permute(0,2,3,1)[water]
    labels=(target[water]-1).long()
    # small extra emphasis on temporary flood within the water-only decision
    weights=torch.tensor([1.0,1.15], device=logits.device)
    return F.cross_entropy(two, labels, weight=weights)

def water_union_dice(logits, target):
    valid=target!=IGNORE_INDEX
    prob=torch.softmax(logits.float(),dim=1)
    p=(prob[:,1]+prob[:,2])[valid]
    g=((target==1)|(target==2)).float()[valid]
    inter=(p*g).sum()
    return 1.0-(2*inter+1.0)/(p.sum()+g.sum()+1.0)

def boundary_mask(target):
    valid=target!=IGNORE_INDEX
    b=torch.zeros_like(valid)
    # mark both sides of valid horizontal/vertical class transitions
    dh = valid[:,:,1:] & valid[:,:,:-1] & (target[:,:,1:] != target[:,:,:-1])
    dw = valid[:,1:,:] & valid[:,:-1,:] & (target[:,1:,:] != target[:,:-1,:])
    b[:,:,1:] |= dh; b[:,:,:-1] |= dh
    b[:,1:,:] |= dw; b[:,:-1,:] |= dw
    return b

def segmentation_loss(output, target):
    logits=output["logits"].float()
    aux=output["aux_logits"].float()

    per_pixel = F.cross_entropy(logits, target, weight=CLASS_WEIGHTS.float(),
                                ignore_index=IGNORE_INDEX, reduction="none")
    valid=target!=IGNORE_INDEX
    ce=per_pixel[valid].mean() if valid.any() else logits.sum()*0.0
    lov=lovasz_softmax_loss(logits,target)
    wt=conditional_water_type_ce(logits,target)
    temp=focal_tversky_class(logits,target,2,alpha=0.60,beta=0.40,gamma=0.75)
    perm=focal_tversky_class(logits,target,1,alpha=0.52,beta=0.48,gamma=0.75)
    water=water_union_dice(logits,target)
    bm=boundary_mask(target)
    bce=per_pixel[bm].mean() if bm.any() else ce*0.0
    aux_up=F.interpolate(aux,size=target.shape[-2:],mode="bilinear",align_corners=False)
    aux_ce=F.cross_entropy(aux_up,target,weight=CLASS_WEIGHTS.float(),ignore_index=IGNORE_INDEX)

    total=(W_CE*ce + W_LOVASZ*lov + W_WTYPE*wt + W_TEMP_TV*temp + W_PERM_TV*perm
           + W_WATER*water + W_BOUNDARY*bce + W_AUX*aux_ce)
    parts={"ce":float(ce.detach()),"lovasz":float(lov.detach()),"water_type":float(wt.detach()),
           "temp_tv":float(temp.detach()),"perm_tv":float(perm.detach()),"water_dice":float(water.detach()),
           "boundary_ce":float(bce.detach()),"aux_ce":float(aux_ce.detach())}
    return total, parts

with torch.inference_mode():
    smoke=model(tiny_sar,tiny_opt,tiny_rel,return_aux=True)
    smoke_target=batch["target"][:1].to(DEVICE)
    smoke_loss, smoke_parts=segmentation_loss(smoke,smoke_target)
print("Loss smoke test:",float(smoke_loss))
print("Components:",smoke_parts)
assert torch.isfinite(smoke_loss)
print("P8-C09 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P8-C10 — METRICS + EVALUATION + TTA
# ============================================================

def confusion_from_predictions(
    pred,
    target,
):
    valid = (
        target
        != IGNORE_INDEX
    )

    target_v = target[
        valid
    ].long()

    pred_v = pred[
        valid
    ].long()

    indices = (
        NUM_CLASSES
        * target_v
        + pred_v
    )

    cm = torch.bincount(
        indices,
        minlength=(
            NUM_CLASSES
            * NUM_CLASSES
        ),
    )

    return cm.reshape(
        NUM_CLASSES,
        NUM_CLASSES,
    )


def metrics_from_confusion(cm):
    cm = (
        cm
        .detach()
        .cpu()
        .numpy()
        .astype(np.float64)
    )

    tp = np.diag(cm)

    fp = (
        cm.sum(axis=0)
        - tp
    )

    fn = (
        cm.sum(axis=1)
        - tp
    )

    denom_iou = (
        tp + fp + fn
    )

    iou = np.divide(
        tp,
        denom_iou,
        out=np.full_like(
            tp,
            np.nan,
            dtype=np.float64,
        ),
        where=(
            denom_iou > 0
        ),
    )

    denom_dice = (
        2.0 * tp
        + fp
        + fn
    )

    dice = np.divide(
        2.0 * tp,
        denom_dice,
        out=np.full_like(
            tp,
            np.nan,
            dtype=np.float64,
        ),
        where=(
            denom_dice > 0
        ),
    )

    precision = np.divide(
        tp,
        tp + fp,
        out=np.full_like(
            tp,
            np.nan,
            dtype=np.float64,
        ),
        where=(
            tp + fp > 0
        ),
    )

    recall = np.divide(
        tp,
        tp + fn,
        out=np.full_like(
            tp,
            np.nan,
            dtype=np.float64,
        ),
        where=(
            tp + fn > 0
        ),
    )

    accuracy = (
        tp.sum()
        / max(
            cm.sum(),
            1.0,
        )
    )

    result = {
        "mIoU": float(
            np.nanmean(iou)
        ),
        "macro_dice": float(
            np.nanmean(dice)
        ),
        "accuracy": float(
            accuracy
        ),
        "confusion": cm.tolist(),
    }

    for i, name in enumerate(
        CLASS_NAMES
    ):
        result[
            f"{name}_IoU"
        ] = float(iou[i])

        result[
            f"{name}_Dice"
        ] = float(dice[i])

        result[
            f"{name}_Precision"
        ] = float(
            precision[i]
        )

        result[
            f"{name}_Recall"
        ] = float(
            recall[i]
        )

    return result


@torch.inference_mode()
def evaluate_model(
    model,
    loader,
):
    model.eval()

    cm = torch.zeros(
        (
            NUM_CLASSES,
            NUM_CLASSES,
        ),
        dtype=torch.long,
        device=DEVICE,
    )

    loss_total = 0.0
    batches = 0

    for batch_eval in loader:
        sar = batch_eval[
            "sar"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        optical = batch_eval[
            "optical"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        reliability = batch_eval[
            "reliability"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        target = batch_eval[
            "target"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        with torch.amp.autocast(
            "cuda",
            enabled=AMP_ENABLED
        ):
            output = model(
                sar,
                optical,
                reliability,
                return_aux=True,
            )

            loss, _ = (
                segmentation_loss(
                    output,
                    target,
                )
            )

        pred = output[
            "logits"
        ].argmax(
            dim=1
        )

        cm += confusion_from_predictions(
            pred,
            target,
        )

        loss_total += float(
            loss.detach()
        )

        batches += 1

    result = metrics_from_confusion(
        cm
    )

    result[
        "loss"
    ] = (
        loss_total
        / max(
            batches,
            1,
        )
    )

    return result


def _tta_transform(
    tensor,
    mode,
):
    if mode == "identity":
        return tensor

    if mode == "hflip":
        return torch.flip(
            tensor,
            dims=[-1],
        )

    if mode == "vflip":
        return torch.flip(
            tensor,
            dims=[-2],
        )

    if mode == "rot180":
        return torch.rot90(
            tensor,
            k=2,
            dims=(-2, -1),
        )

    raise ValueError(mode)


def _tta_inverse(
    tensor,
    mode,
):
    # All four selected transforms are self-inverse.
    return _tta_transform(
        tensor,
        mode,
    )


@torch.inference_mode()
def evaluate_model_tta(
    model,
    loader,
):
    model.eval()

    modes = [
        "identity",
        "hflip",
        "vflip",
        "rot180",
    ]

    cm = torch.zeros(
        (
            NUM_CLASSES,
            NUM_CLASSES,
        ),
        dtype=torch.long,
        device=DEVICE,
    )

    for batch_eval in loader:
        sar = batch_eval[
            "sar"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        optical = batch_eval[
            "optical"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        reliability = batch_eval[
            "reliability"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        target = batch_eval[
            "target"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        logits_sum = None

        for mode in modes:
            sar_t = _tta_transform(
                sar,
                mode,
            )

            opt_t = _tta_transform(
                optical,
                mode,
            )

            rel_t = _tta_transform(
                reliability,
                mode,
            )

            with torch.cuda.amp.autocast(
                enabled=AMP_ENABLED
            ):
                logits = model(
                    sar_t,
                    opt_t,
                    rel_t,
                    return_aux=False,
                )

            logits = _tta_inverse(
                logits,
                mode,
            )

            if logits_sum is None:
                logits_sum = (
                    logits.float()
                )
            else:
                logits_sum += (
                    logits.float()
                )

        logits_mean = (
            logits_sum
            / len(modes)
        )

        pred = logits_mean.argmax(
            dim=1
        )

        cm += confusion_from_predictions(
            pred,
            target,
        )

    return metrics_from_confusion(
        cm
    )


def compact_metrics(m):
    return {
        "mIoU": round(
            m[
                "mIoU"
            ],
            4,
        ),
        "non_flood": round(
            m[
                "non_flood_IoU"
            ],
            4,
        ),
        "permanent": round(
            m[
                "permanent_water_IoU"
            ],
            4,
        ),
        "temporary": round(
            m[
                "temporary_flood_IoU"
            ],
            4,
        ),
    }


print("P8-C10 COMPLETE ✅")


In [ ]:
# ============================================================
# CELL: P8-C11 — VERIFY WARM-START VALIDATION BEFORE TRAINING
# ============================================================
# This is a critical reproducibility guard: zero-init 04F must reproduce
# the archived 04E single-view validation (~0.7743 mIoU).

warm_val = evaluate_model(model, val_loader)
print("Warm-start validation:", compact_metrics(warm_val))

REFERENCE_NO_TTA_MIOU = 0.7743
if abs(warm_val["mIoU"] - REFERENCE_NO_TTA_MIOU) > 0.01:
    raise RuntimeError(
        f"04E reproduction check failed: got {warm_val['mIoU']:.4f}, expected near {REFERENCE_NO_TTA_MIOU:.4f}. "
        "Stop before training and inspect checkpoint/data inputs."
    )
print("✅ 04E warm-start reproduced within tolerance")


In [ ]:
# ============================================================
# CELL: P8-C12 — CONSERVATIVE 3-STAGE FINE-TUNING
# ============================================================
MAX_EPOCHS = 22
STAGE_A_END = 4
STAGE_B_END = 12
EARLY_STOPPING_PATIENCE = 6
GRAD_CLIP_NORM = 0.7
WEIGHT_DECAY = 1e-2

LR_REFINER_A = 5e-4
LR_REFINER_B = 2e-4
LR_BASE_B    = 3e-5
LR_REFINER_C = 1e-4
LR_BASE_C    = 1e-5
LR_BACKBONE_C= 6e-7

HISTORY_PATH = HISTORY_ROOT / "04f_history.csv"
BEST_CHECKPOINT_PATH = CHECKPOINT_ROOT / "FloodShift_04F_best.pt"


def set_all_requires_grad(module, flag):
    for p in module.parameters(): p.requires_grad = flag

def unfreeze_last_tm_block(base_model, last_n=1, depth=12):
    for p in base_model.backbone.parameters(): p.requires_grad=False
    matched=[]; min_block=depth-last_n
    for name,p in base_model.backbone.named_parameters():
        m=re.search(r"(?:^|\.)encoder\.(\d+)\.",name)
        if m and int(m.group(1))>=min_block:
            p.requires_grad=True; matched.append(name)
        if "encoder_norm" in name:
            p.requires_grad=True; matched.append(name)
    if not matched: raise RuntimeError("Could not find TerraMind final block names.")
    return matched

def configure_stage(stage):
    # Freeze everything first.
    set_all_requires_grad(model.base, False)
    set_all_requires_grad(model.refiner, True)
    groups=[]
    if stage=="A":
        groups=[{"params":model.refiner.parameters(),"lr":LR_REFINER_A,"name":"refiner"}]
    elif stage=="B":
        # No TerraMind blocks yet. Adapt only the task-specific tail and fusion corrections.
        selected=[model.base.routers, model.base.deep_rift, model.base.decoder1,
                  model.base.final_refine, model.base.segmentation_head, model.base.aux_head]
        for m in selected: set_all_requires_grad(m,True)
        base_params=[p for m in selected for p in m.parameters() if p.requires_grad]
        groups=[{"params":model.refiner.parameters(),"lr":LR_REFINER_B,"name":"refiner"},
                {"params":base_params,"lr":LR_BASE_B,"name":"base_tail"}]
    elif stage=="C":
        selected=[model.base.routers, model.base.deep_rift, model.base.decoder2, model.base.decoder1,
                  model.base.final_refine, model.base.segmentation_head, model.base.aux_head]
        for m in selected: set_all_requires_grad(m,True)
        matched=unfreeze_last_tm_block(model.base,last_n=1,depth=12)
        print("Stage C TerraMind trainable tensors:",len(matched))
        base_tail=[]
        for m in selected: base_tail += [p for p in m.parameters() if p.requires_grad]
        bb=[p for p in model.base.backbone.parameters() if p.requires_grad]
        groups=[{"params":model.refiner.parameters(),"lr":LR_REFINER_C,"name":"refiner"},
                {"params":base_tail,"lr":LR_BASE_C,"name":"base_tail"},
                {"params":bb,"lr":LR_BACKBONE_C,"name":"tm_last_block"}]
    else: raise ValueError(stage)
    # Deduplicate parameters defensively.
    seen=set(); clean=[]
    for g in groups:
        ps=[]
        for p in g["params"]:
            if id(p) not in seen: seen.add(id(p)); ps.append(p)
        clean.append({**g,"params":ps})
    opt=torch.optim.AdamW(clean,weight_decay=WEIGHT_DECAY)
    return opt

def scheduler_for(opt, epochs):
    return torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=max(epochs,1),eta_min=1e-7)

def save_ckpt(path, epoch, stage, metrics):
    torch.save({"state_dict":model.state_dict(),"epoch":int(epoch),"stage":stage,
                "val_metrics":metrics,"architecture":"FloodShift-04F-HierarchicalRefiner",
                "warmstart_sha256":warm_sha,"target_semantics":CLASS_NAMES},path)

# Preserve exact warm start as epoch 0 fallback.
best_val=copy.deepcopy(warm_val); best_epoch=0; best_stage="warmstart"
save_ckpt(BEST_CHECKPOINT_PATH,0,"warmstart",best_val)
history=[]; no_improve=0; nonfinite_batches_total=0
stage="A"; optimizer=configure_stage(stage); scheduler=scheduler_for(optimizer,STAGE_A_END)
scaler=torch.amp.GradScaler("cuda",enabled=AMP_ENABLED)
start_all=time.time()

for epoch in range(1,MAX_EPOCHS+1):
    if epoch==STAGE_A_END+1:
        stage="B"; optimizer=configure_stage(stage); scheduler=scheduler_for(optimizer,STAGE_B_END-STAGE_A_END); no_improve=0
        print("\n"+"="*112+"\nSTAGE B — refiner + task tail; TerraMind still frozen\n"+"="*112)
    if epoch==STAGE_B_END+1:
        stage="C"; optimizer=configure_stage(stage); scheduler=scheduler_for(optimizer,MAX_EPOCHS-STAGE_B_END); no_improve=0
        print("\n"+"="*112+"\nSTAGE C — final TerraMind block only, ultra-low LR\n"+"="*112)

    model.train(); model.base.backbone.eval()
    running=0.0; nloss=0; skipped=0
    optimizer.zero_grad(set_to_none=True); t0=time.time()

    for step,b in enumerate(train_loader,start=1):
        sar=b["sar"].to(DEVICE,non_blocking=True); opt=b["optical"].to(DEVICE,non_blocking=True)
        rel=b["reliability"].to(DEVICE,non_blocking=True); target=b["target"].to(DEVICE,non_blocking=True)
        with torch.amp.autocast("cuda",enabled=AMP_ENABLED):
            out=model(sar,opt,rel,return_aux=True)
        # Loss deliberately outside autocast -> FP32 numerics.
        loss,parts=segmentation_loss(out,target)
        if not torch.isfinite(loss):
            skipped += 1; optimizer.zero_grad(set_to_none=True); continue
        scaled=loss/GRAD_ACCUM_STEPS
        scaler.scale(scaled).backward()
        do_step=(step%GRAD_ACCUM_STEPS==0 or step==len(train_loader))
        if do_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],GRAD_CLIP_NORM)
            scaler.step(optimizer); scaler.update(); optimizer.zero_grad(set_to_none=True)
        running += float(loss.detach()); nloss += 1

    nonfinite_batches_total += skipped
    train_loss=running/max(nloss,1)
    val=evaluate_model(model,val_loader)
    elapsed=time.time()-t0
    row={"epoch":epoch,"stage":stage,"train_loss":train_loss,"skipped_nonfinite":skipped,
         "val_loss":val["loss"],"val_mIoU":val["mIoU"],
         "val_non_flood_IoU":val["non_flood_IoU"],"val_permanent_water_IoU":val["permanent_water_IoU"],
         "val_temporary_flood_IoU":val["temporary_flood_IoU"],"epoch_seconds":elapsed}
    history.append(row); pd.DataFrame(history).to_csv(HISTORY_PATH,index=False)
    print(f"Epoch {epoch:02d} [{stage}] | train {train_loss:.4f} | val {val['loss']:.4f} | "
          f"mIoU {val['mIoU']:.4f} | perm {val['permanent_water_IoU']:.4f} | temp {val['temporary_flood_IoU']:.4f} | "
          f"skip {skipped} | {elapsed/60:.1f} min")

    better=(val["mIoU"] > best_val["mIoU"] + 1e-7 or
            (abs(val["mIoU"]-best_val["mIoU"])<=1e-7 and val["temporary_flood_IoU"]>best_val["temporary_flood_IoU"]))
    if better:
        best_val=copy.deepcopy(val); best_epoch=epoch; best_stage=stage; no_improve=0
        save_ckpt(BEST_CHECKPOINT_PATH,epoch,stage,val); print("  ✅ NEW BEST")
    else:
        no_improve += 1
    scheduler.step()

    if epoch > STAGE_B_END and no_improve >= EARLY_STOPPING_PATIENCE:
        print("Early stopping in Stage C."); break
    gc.collect();
    if torch.cuda.is_available(): torch.cuda.empty_cache()

training_minutes=(time.time()-start_all)/60
best_payload=torch.load(BEST_CHECKPOINT_PATH,map_location=DEVICE,weights_only=False)
model.load_state_dict(best_payload["state_dict"],strict=True)
best_val=best_payload["val_metrics"]; best_epoch=int(best_payload["epoch"]); best_stage=best_payload["stage"]
print("\n"+"="*112)
print("04F TRAINING COMPLETE")
print("Best epoch/stage :",best_epoch,best_stage)
print("Best validation  :",compact_metrics(best_val))
print("Training minutes :",f"{training_minutes:.1f}")
print("Non-finite batches skipped:",nonfinite_batches_total)


In [ ]:
# ============================================================
# CELL: P8-C13 — VALIDATION-ONLY TTA SELECTION + STRICT GATE
# ============================================================
TTA_MIN_GAIN=0.001
PROMOTION_MIN_MIOU=0.790
PROMOTION_MIN_TEMP_IOU=0.570
PROMOTION_MIN_PERM_IOU=0.820

chosen_val=best_val
print("Evaluating 4-way TTA on validation...")
tta_val=evaluate_model_tta(model,val_loader)
print("No-TTA:",compact_metrics(chosen_val))
print("TTA   :",compact_metrics(tta_val))
use_tta=tta_val["mIoU"] >= chosen_val["mIoU"] + TTA_MIN_GAIN
final_val=tta_val if use_tta else chosen_val
inference_mode="4way_tta" if use_tta else "single_view"
promotion_pass=(final_val["mIoU"]>=PROMOTION_MIN_MIOU and
                final_val["temporary_flood_IoU"]>=PROMOTION_MIN_TEMP_IOU and
                final_val["permanent_water_IoU"]>=PROMOTION_MIN_PERM_IOU)
print("\n"+"="*112)
print("P8-C13 — 04F PROMOTION GATE")
print("="*112)
print("Selected inference:",inference_mode)
print("Final validation  :",compact_metrics(final_val))
print(f"Gate: mIoU>={PROMOTION_MIN_MIOU:.3f}, temp>={PROMOTION_MIN_TEMP_IOU:.3f}, perm>={PROMOTION_MIN_PERM_IOU:.3f}")
print("PROMOTION:","PASS ✅" if promotion_pass else "FAIL — FROZEN TEST REMAINS LOCKED 🔒")


In [ ]:
# ============================================================
# CELL: P8-C14 — FROZEN GEOGRAPHIC TEST (MANUAL UNLOCK)
# ============================================================
# IMPORTANT: leave False until P8-C13 prints PROMOTION: PASS.
# Then change ONLY this flag to True and rerun this cell once.
UNLOCK_FROZEN_TEST = False

test_metrics=None
if not promotion_pass:
    print("🔒 Test locked: validation promotion gate failed.")
elif not UNLOCK_FROZEN_TEST:
    print("✅ Gate passed, but test is still manually locked.")
    print("Set UNLOCK_FROZEN_TEST=True and rerun ONLY P8-C14 when ready.")
else:
    print("Running frozen Pakistan+Spain geographic test...")
    test_metrics=evaluate_model_tta(model,test_loader) if use_tta else evaluate_model(model,test_loader)
    print("\n"+"="*112)
    print("04F FROZEN GEOGRAPHIC TEST")
    print("="*112)
    print(compact_metrics(test_metrics))
    with open(METRIC_ROOT/"04f_test_metrics.json","w") as f: json.dump(test_metrics,f,indent=2)
    baseline={"mIoU":0.7162,"permanent_water_IoU":0.6183,"temporary_flood_IoU":0.5675}
    print("\n04E reference:",baseline)
    print("Δ mIoU:",test_metrics["mIoU"]-baseline["mIoU"])
    print("Δ permanent:",test_metrics["permanent_water_IoU"]-baseline["permanent_water_IoU"])
    print("Δ temporary:",test_metrics["temporary_flood_IoU"]-baseline["temporary_flood_IoU"])


In [ ]:
# ============================================================
# CELL: P8-C15 — VALIDATION TEMPERATURE CALIBRATION
#
# Calibration NEVER changes the segmentation mask used for
# model selection. It is post-hoc probability calibration only.
# ============================================================

MAX_CALIBRATION_PIXELS = 200_000


@torch.inference_mode()
def collect_calibration_samples(
    model,
    loader,
    max_pixels=MAX_CALIBRATION_PIXELS,
):
    model.eval()

    logits_cpu = []
    labels_cpu = []

    rng = np.random.default_rng(
        SEED
    )

    for batch_cal in loader:
        sar = batch_cal[
            "sar"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        optical = batch_cal[
            "optical"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        reliability = batch_cal[
            "reliability"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        target = batch_cal[
            "target"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        # Calibration uses selected inference without TTA
        # to keep collection tractable. This calibrates the
        # underlying model probability scale.
        logits = model(
            sar,
            optical,
            reliability,
            return_aux=False,
        ).float()

        valid = (
            target
            != IGNORE_INDEX
        )

        logits_v = (
            logits
            .permute(
                0,
                2,
                3,
                1,
            )[
                valid
            ]
        )

        target_v = target[
            valid
        ]

        if (
            logits_v.shape[0]
            > 4096
        ):
            indices = rng.choice(
                logits_v.shape[0],
                size=4096,
                replace=False,
            )

            indices = torch.tensor(
                indices,
                device=logits_v.device,
                dtype=torch.long,
            )

            logits_v = logits_v[
                indices
            ]

            target_v = target_v[
                indices
            ]

        logits_cpu.append(
            logits_v.cpu()
        )

        labels_cpu.append(
            target_v.cpu()
        )

        current = sum(
            x.shape[0]
            for x in logits_cpu
        )

        if current >= max_pixels:
            break

    logits_all = torch.cat(
        logits_cpu,
        dim=0,
    )[
        :max_pixels
    ]

    labels_all = torch.cat(
        labels_cpu,
        dim=0,
    )[
        :max_pixels
    ]

    return (
        logits_all,
        labels_all,
    )


cal_logits, cal_labels = (
    collect_calibration_samples(
        model,
        val_loader,
    )
)

log_temperature = torch.zeros(
    1,
    device=DEVICE,
    requires_grad=True,
)

cal_logits_gpu = cal_logits.to(
    DEVICE
)

cal_labels_gpu = cal_labels.to(
    DEVICE
)

optimizer_temp = torch.optim.LBFGS(
    [log_temperature],
    lr=0.05,
    max_iter=50,
)


def temp_closure():
    optimizer_temp.zero_grad()

    temperature = (
        log_temperature.exp()
        .clamp(
            0.05,
            20.0,
        )
    )

    loss = F.cross_entropy(
        cal_logits_gpu
        / temperature,
        cal_labels_gpu,
    )

    loss.backward()

    return loss


optimizer_temp.step(
    temp_closure
)

temperature = float(
    log_temperature
    .exp()
    .detach()
    .clamp(
        0.05,
        20.0,
    )
    .cpu()
)

calibration_record = {
    "temperature": temperature,
    "calibration_split": "validation_only",
    "n_pixels": int(
        cal_labels.shape[0]
    ),
    "note":
        "post-hoc probability calibration; "
        "not used for checkpoint selection",
}

with open(
    CALIBRATION_ROOT
    / "04f_temperature_scaling.json",
    "w",
) as f:
    json.dump(
        calibration_record,
        f,
        indent=2,
    )

print(
    "Temperature:",
    temperature,
)

print(
    "Calibration pixels:",
    len(cal_labels),
)

print("P8-C15 COMPLETE ✅")

del (
    cal_logits_gpu,
    cal_labels_gpu,
    cal_logits,
    cal_labels,
)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# ============================================================
# CELL: P8-C16 — FINAL EXPORT / ARCHIVE
# ============================================================
protocol={
    "model":"FloodShift-04F-HierarchicalRefiner",
    "warmstart":"FloodShift-FINAL-TerraMind-RIFT",
    "warmstart_sha256":warm_sha,
    "data":{"dataset":"Sen1Floods11 v1.1 HandLabeled","train":314,"validation":68,"test":58,
            "target":"0 background, 1 permanent water, 2 temporary flood, 255 ignore",
            "train_sampling":"fixed class-aware stochastic crops + scene oversampling",
            "test_used_for_training_or_validation":False},
    "refiner":"zero-init hierarchical residual: water-vs-background + temp-vs-permanent",
    "loss":{"CE":W_CE,"Lovasz":W_LOVASZ,"conditional_water_type_CE":W_WTYPE,"temp_focal_tversky":W_TEMP_TV,
            "perm_focal_tversky":W_PERM_TV,"water_union_dice":W_WATER,"boundary_CE":W_BOUNDARY,"aux_CE":W_AUX},
    "training":{"max_epochs":MAX_EPOCHS,"stage_A":"refiner only","stage_B":"refiner + task tail; TerraMind frozen",
                "stage_C":"final TerraMind block only + task tail","full_backbone_unfreeze":False,
                "skipped_nonfinite_batches":nonfinite_batches_total,"minutes":training_minutes},
    "selection":{"best_epoch":best_epoch,"best_stage":best_stage,"best_no_tta":best_val,"inference_mode":inference_mode,
                 "final_validation":final_val,"promotion_pass":bool(promotion_pass),
                 "gate":{"mIoU":PROMOTION_MIN_MIOU,"temporary":PROMOTION_MIN_TEMP_IOU,"permanent":PROMOTION_MIN_PERM_IOU}},
    "test_metrics":test_metrics,
    "temperature_scaling":calibration_record,
}
PROTO=OUTPUT_ROOT/"FloodShift_04F_protocol.json"
with open(PROTO,"w") as f: json.dump(protocol,f,indent=2)

# Checkpoint finite audit + SHA256
final_payload=torch.load(BEST_CHECKPOINT_PATH,map_location="cpu",weights_only=False)
bad=[n for n,t in final_payload["state_dict"].items() if torch.is_floating_point(t) and not torch.isfinite(t).all()]
assert not bad,f"Non-finite final tensors: {bad}"
sha=hashlib.sha256()
with open(BEST_CHECKPOINT_PATH,"rb") as f:
    for chunk in iter(lambda:f.read(1024*1024),b""): sha.update(chunk)
final_sha=sha.hexdigest()
protocol["checkpoint_integrity"]={"sha256":final_sha,"all_finite":True}
with open(PROTO,"w") as f: json.dump(protocol,f,indent=2)

# Archive the minimal reproducible outputs.
ARCHIVE=Path("/kaggle/working/FloodShift_04F_ARCHIVE.zip")
with zipfile.ZipFile(ARCHIVE,"w",compression=zipfile.ZIP_STORED) as z:
    for p in [BEST_CHECKPOINT_PATH,PROTO,HISTORY_PATH,CALIBRATION_ROOT/"04f_temperature_scaling.json"]:
        if Path(p).exists(): z.write(p,arcname=Path(p).name)

print("="*112)
print("P8-C16 — FINAL EXPORT")
print("="*112)
print("Checkpoint:",BEST_CHECKPOINT_PATH)
print("SHA256    :",final_sha)
print("Protocol  :",PROTO)
print("History   :",HISTORY_PATH)
print("Archive   :",ARCHIVE)
print("Validation:",compact_metrics(final_val))
print("Promotion :",promotion_pass)
print("Test      :",compact_metrics(test_metrics) if test_metrics else "LOCKED / NOT RUN")


## Execution order

1. Attach the **Notebook-03 frozen preprocessing output** and either the extracted 04E files or `FloodShift_FINAL_ARCHIVE.zip`.
2. GPU T4, Internet ON.
3. Run **P8-C00**. If the next cell hits NumPy/Pandas ABI mismatch, restart the Python kernel once, **do not rerun C00**, then continue.
4. Run **P8-C01 → C13**.
5. If and only if C13 prints `PROMOTION: PASS`, set `UNLOCK_FROZEN_TEST=True` in **C14** and run C14 once.
6. Run C15 and C16.

### Success interpretation
The notebook cannot guarantee `0.75+` test mIoU. It is designed to maximize a *scientifically controlled* final attempt without test-driven tuning. If it does not improve enough, segmentation development stops and the project moves to the next pipeline stage rather than repeatedly adapting to the same test set.
